# Hello World – PySpark

Ce notebook vérifie que l'environnement Docker + PySpark + Jupyter est opérationnel.

**Prérequis :** lancer via `docker compose up` depuis la racine du projet.

In [ ]:
import sys
print(f"Python {sys.version}")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("BGES – Hello World Notebook")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"PySpark version : {spark.version}")
java_ver = spark.sparkContext._jvm.System.getProperty("java.version")
print(f"Java version    : {java_ver}")

## Création d'un DataFrame de test

In [ ]:
data = [
    ("Paris",       "France",   2_161_000),
    ("Berlin",      "Germany",  3_645_000),
    ("London",      "UK",       8_982_000),
    ("New York",    "USA",      8_336_000),
    ("Los Angeles", "USA",      3_979_000),
    ("Shanghai",    "China",   24_870_000),
]

df = spark.createDataFrame(data, ["city", "country", "population"])
df.printSchema()

## Affichage et agrégation

In [ ]:
df.show()

In [ ]:
df.groupBy("country") \
  .agg(F.sum("population").alias("total_population")) \
  .orderBy(F.desc("total_population")) \
  .show()

## Visualisation avec pandas + matplotlib

In [ ]:
import matplotlib.pyplot as plt

pdf = df.toPandas().sort_values("population", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(pdf["city"], pdf["population"] / 1_000_000, color="steelblue")
ax.set_xlabel("Population (millions)")
ax.set_title("Villes du projet BGES")
plt.tight_layout()
plt.show()

In [ ]:
spark.stop()
print("Session Spark arrêtée.")